# 01 - Thu thập dữ liệu tử vong Việt Nam

**Mục tiêu:** tải và kiểm tra dữ liệu từ UN WPP (nguồn chính), đối chiếu GSO.

**Checklist:**
- Tải bảng sống tuổi đơn UN WPP (1950–nay) vào `data/raw/`
- Tải bảng sống rút gọn GSO (2009, 2019) vào `data/external/`
- Kiểm tra tính liên tục theo năm, missing values
- Chạy `python -m src.data.make_dataset` để dựng Dxt, Ext
- Ghi chú nguồn + ngày tải vào `data/external/SOURCES.md`

## Tải dữ liệu Bảng sống từ UN WPP

Tải thủ công 3 file "Single age life table estimates" tại https://population.un.org/wpp/ (mục *Life Tables*):

| Giới tính | Tên file (đặt vào `data/raw/`) |
|---|---|
| Cả hai giới | `WPP2024_MORT_F06_1_SINGLE_AGE_LIFE_TABLE_ESTIMATES_BOTH_SEXES.xlsx` |
| Nam | `WPP2024_MORT_F06_2_SINGLE_AGE_LIFE_TABLE_ESTIMATES_MALE.xlsx` |
| Nữ | `WPP2024_MORT_F06_3_SINGLE_AGE_LIFE_TABLE_ESTIMATES_FEMALE.xlsx` |

Ghi lại ngày tải vào `data/external/SOURCES.md`. Bảng sống GSO (đối chiếu, không dùng để fit) số hoá thủ công từ báo cáo TĐT thành CSV với cột tối thiểu `age`, `sex`, `mx` hoặc `qx`, đặt vào `data/external/`.

In [12]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from src.config import DATA_RAW, DATA_PROCESSED, DATA_INTERIM, load_params
from src.data.make_dataset import WPP_FILES
import pandas as pd

cfg = load_params()
list_files = [f for f in WPP_FILES.values() if (DATA_RAW / f).exists()]
if len(list_files) == 3:
    print("Xác nhận đủ 3 file từ WPP2024")
    for f in list_files:
        print(f)
else:
    print("Thiếu file!")

print("Đọc thử file tổng: WPP2024_MORT_F06_1_SINGLE_AGE_LIFE_TABLE_ESTIMATES_BOTH_SEXES.xlsx")
df = pd.read_excel(DATA_RAW / 'WPP2024_MORT_F06_1_SINGLE_AGE_LIFE_TABLE_ESTIMATES_BOTH_SEXES.xlsx', skiprows=16)
df.head()

Xác nhận đủ 3 file từ WPP2024
WPP2024_MORT_F06_1_SINGLE_AGE_LIFE_TABLE_ESTIMATES_BOTH_SEXES.xlsx
WPP2024_MORT_F06_2_SINGLE_AGE_LIFE_TABLE_ESTIMATES_MALE.xlsx
WPP2024_MORT_F06_3_SINGLE_AGE_LIFE_TABLE_ESTIMATES_FEMALE.xlsx
Đọc thử file tổng: WPP2024_MORT_F06_1_SINGLE_AGE_LIFE_TABLE_ESTIMATES_BOTH_SEXES.xlsx


,Index,Variant,"Region, subregion, country or area *",Notes,Location code,ISO3 Alpha-code,ISO2 Alpha-code,SDMX code**,Type,Parent code,...,"Central death rate m(x,n)","Probability of dying q(x,n)","Probability of surviving p(x,n)",Number of survivors l(x),"Number of deaths d(x,n)","Number of person-years lived L(x,n)","Survival ratio S(x,n)",Person-years lived T(x),Expectation of life e(x),"Average number of years lived a(x,n)"
0,1,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,0.152936,0.138135,0.861865,100000,13813.493,90322.011,0.90322,4639439.562,46.3944,0.299381
1,2,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,0.049267,0.048082,0.951918,86186.507,4144.044,84114.485,0.931273,4549117.552,52.7822,0.5
2,3,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,0.028588,0.028185,0.971815,82042.463,2312.376,80886.276,0.961621,4465003.066,54.4231,0.5
3,4,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,0.01839,0.018222,0.981778,79730.088,1452.854,79003.661,0.976725,4384116.791,54.987,0.5
4,5,Estimates,World,NaN,900,NaN,NaN,1.0,World,0,...,0.012805,0.012723,0.987277,78277.234,995.954,77779.257,0.984502,4305113.13,54.9983,0.5


**Các thông tin quan trọng cần dùng cho fit model:**
- Số người tử vong quan sát được ỏ độ tuổi $x$ trong $t$ năm - $D_{x, t}$
- Số người ở độ tuổi $x$ chịu rủi ro trong $t$ năm (`Exposure`) - $E_{x, t}$
- Tuổi - Age
- Năm - Year

Tuy nhiên, trong file không có $E_{x,t}$. Do đó, `Exposure` sẽ lấy xấp xỉ từ L(x,t) - person-years lived trong bảng sống - vì file life table của UN không tách deaths/exposure dân số thực. 

Do đó $D_{x,t} = m_x * E_{x,t}$ là MỘT GIẢ ĐỊNH, không phải số ca tử vong thực tế đếm được.

In [20]:
from src.data.make_dataset import main as build_dataset

missing = [f for f in WPP_FILES.values() if not (DATA_RAW / f).exists()]
if missing:
    print("Còn thiếu trong data/raw/:")
    for f in missing:
        print(" -", f)
else:
    print("Đã có đủ 3 file UN WPP, dựng Dxt/Ext...")
    build_dataset()

Đã có đủ 3 file UN WPP, dựng Dxt/Ext...
[ok] male: 101 tuổi x 69 năm -> data/processed/*_male.csv
[ok] female: 101 tuổi x 69 năm -> data/processed/*_female.csv
[ok] total: 101 tuổi x 69 năm -> data/processed/*_total.csv


## Thu thập dữ liệu Bảng sống từ Báo cáo Tổng điểu tra dân số 2019
Tải báo cáo Tổng điều tra dân số 2019 từ . Xác định Bảng sống theo giới tính được ghi nhận ở `Biểu 6.7`, `trang 96` trong báo cáo. Tuy nhiên, báo cáo được định dạng PDF, để có thể sử dụng, cần lấy dữ liệu theo định dạnh CSV.

| Dữ liệu | Tên file|
|---|---|
| Báo cáo TĐT Dân số 2019 | `TDT-Dan-so-2019-1.pdf` |
| Dữ liệu Bảng sống theo giới tính trích xuất từ TĐT | `gso_bang_song_tdt2019.csv` |

Dữ liệu này KHÔNG dùng để fit mô hình (chuỗi năm quá ngắn/thưa), chỉ để kiểm chứng chéo với UN WPP ở notebook EDA.

In [23]:
from src.config import DATA_INTERIM

df_gso_life_table = pd.read_csv(DATA_INTERIM / 'gso_bang_song_tdt2019.csv')
df_gso_life_table.head()

,sex,x,n,nLx,lx,ndx,nqx,npx,nmx,Tx,ex
0,male,0,1.0,98006,100000,1584,0.0158,0.9842,0.0162,7101496,71.0
1,male,1,4.0,390225,98416,1148,0.0117,0.9883,0.0029,7003490,71.2
2,male,5,5.0,486438,97268,203,0.0021,0.9979,0.0004,6613266,68.0
3,male,10,5.0,485465,97065,296,0.0031,0.9969,0.0006,6126827,63.1
4,male,15,5.0,484066,96769,449,0.0046,0.9954,0.0009,5641362,58.3
